# TAAS 교통사고 위치정보 수집 노트북

이 노트북은 TAAS GIS 화면이 사용하는 요청을 재현하여 교통사고 목록과 위치정보를 JSON/CSV로 저장합니다. 작성 참고: [TAAS에서 교통사고 위치정보 가져오기](https://yvelkram.tistory.com/2)

> **중요**: 아래 주소는 공식 공개 API가 아니라 TAAS 웹 화면의 내부 요청입니다. TAAS 화면이나 응답 구조가 바뀌면 코드 수정이 필요할 수 있습니다. TAAS 이용정책을 확인하고, 서버에 부담을 주는 반복·대량 요청은 피하세요. 연구 결과에는 데이터 출처와 TAAS 링크를 명시하는 것이 좋습니다.

이 노트북은 로그인 우회나 보안장치 무력화를 하지 않습니다. 사용자가 정상적으로 연 TAAS 브라우저 세션의 값 두 개를 실행 시점에만 입력받습니다. 입력값은 파일로 저장하지 않습니다.

## 사용 순서

1. 아래 **사용자 설정** 셀에서 시작 일시, 종료 일시, 법정동 코드, 사고 심각도를 바꿉니다.
2. TAAS GIS를 Chrome에서 열고 실제로 한 번 조회합니다.
3. 개발자 도구에서 `TAASJSESSIONID`와 `x-csrf-token`을 확인합니다.
4. **보안값 입력** 셀을 실행하고 두 값을 붙여넣습니다. 화면에는 입력값이 표시되지 않습니다.
5. 아래 셀을 차례로 실행합니다. 결과는 현재 작업 폴더의 `taas_output` 폴더에 저장됩니다.

TAAS 요청 자체는 글에서 설명한 것처럼 **연도 단위**입니다. 이 노트북은 입력한 시작·종료 일시가 연도 전체보다 좁으면, 내려받은 결과에서 사고일시 열을 찾아 정확한 기간으로 한 번 더 필터링합니다.

## Chrome 개발자 도구에서 필요한 값 찾기

1. Chrome에서 [TAAS 교통사고 GIS](https://taas.koroad.or.kr/gis/mcm/mcl/initMap.do?menuId=GIS_GMP_STS_RSN)를 엽니다.
2. `F12`를 누르거나, 페이지에서 마우스 오른쪽 버튼 → **검사**를 눌러 개발자 도구를 엽니다.
3. 상단의 **Network(네트워크)** 탭으로 이동합니다. 기록이 복잡하면 지우기 아이콘을 눌러 목록을 비웁니다.
4. TAAS 화면에서 작은 지역과 짧은 기간을 선택하고 **검색/조회** 버튼을 한 번 누릅니다.
5. Network 필터에 `selectAccidentInfo.do`를 입력하고 해당 요청을 클릭합니다. 보이지 않으면 Network 탭을 연 상태에서 다시 조회합니다.
6. **Headers → Request Headers**에서 다음 두 값을 찾습니다.
   - `Cookie` 안의 `TAASJSESSIONID=...` 값: `=` 뒤부터 다음 `;` 전까지 복사합니다. 실수로 `TAASJSESSIONID=`까지 붙여넣어도 아래 코드가 정리합니다.
   - `x-csrf-token: ...` 값: 콜론 뒤의 토큰 전체를 복사합니다. 헤더 이름까지 붙여넣어도 아래 코드가 정리합니다.
7. TAAS 탭을 닫거나 오래 기다리면 세션이 만료될 수 있습니다. 그때는 화면을 새로 열어 다시 조회하고 두 값을 새로 가져옵니다.

> **보안 주의**: 세션 ID와 CSRF 토큰은 다른 사람에게 보내거나 캡처 화면에 노출하지 마세요. 로그인 가능한 서비스에서는 세션 탈취가 계정 접근으로 이어질 수 있습니다. 이 노트북에도 문자열로 직접 적지 말고 `getpass()` 입력 셀을 사용하세요.

In [1]:
from __future__ import annotations

import json
import re
from getpass import getpass
from pathlib import Path
from typing import Any

try:
    import pandas as pd
    import requests
    from IPython.display import display
except ImportError as exc:
    raise ImportError(
        "필요 패키지가 없습니다. 새 셀에서 `%pip install pandas requests`를 실행한 뒤 커널을 다시 시작하세요."
    ) from exc

print(f"pandas {pd.__version__} / requests {requests.__version__}")

pandas 3.0.2 / requests 2.32.5


## 사용자 설정

보통 아래 셀만 수정하면 됩니다.

- `START_DATETIME`, `END_DATETIME`: `YYYY`, `YYYY-MM`, `YYYY-MM-DD`, `YYYY-MM-DD HH:MM:SS` 형식을 사용할 수 있습니다. 종료값을 날짜까지만 쓰면 그날 23:59:59까지 포함합니다.
- `LEGAL_DONG_CODE`: `%` 없이 입력합니다. 예: 서울 전체 `11`, 전주시 전체 `4511`, 전주시 덕진구 `45113`. TAAS 화면의 실제 요청 Payload에서 `legaldongCode`를 확인하는 방법이 가장 확실합니다.
- `ACCIDENT_TYPES`: `01` 사망, `02` 중상, `03` 경상, `04` 부상신고입니다. 필요한 값만 남길 수 있습니다.
- `DATETIME_COLUMN`, `DATE_COLUMN`, `TIME_COLUMN`: 자동 탐지가 실패할 때만 결과 열 이름을 직접 적는 고급 설정입니다. 날짜와 시간이 한 열이면 `DATETIME_COLUMN`, 서로 다른 열이면 `DATE_COLUMN`과 `TIME_COLUMN`을 사용합니다.
- `APPLY_EXACT_DATETIME_FILTER=False`로 바꾸면 서버에서 받은 연도 전체 결과를 그대로 저장합니다.

In [2]:
# =========================
# 자주 바꾸는 설정
# =========================
START_DATETIME = "2020-01-01 00:00:00"
END_DATETIME = "2024-12-31 23:59:59"
LEGAL_DONG_CODE = "2818510600" 
ACCIDENT_TYPES = ["01", "02", "03", "04"]

# =========================
# 필요할 때만 바꾸는 설정
# =========================
APPLY_EXACT_DATETIME_FILTER = True
DATETIME_COLUMN = None  # 예: "acdntDate" (날짜와 시간이 한 열에 있을 때)
DATE_COLUMN = None      # 예: "acdntYmd" (날짜 열이 따로 있을 때)
TIME_COLUMN = None      # 예: "acdntTime" (시간 열이 따로 있을 때)

OUTPUT_DIRECTORY = "taas_output"
OUTPUT_BASENAME = "taas_accidents"
REQUEST_TIMEOUT_SECONDS = 120

In [4]:
ACCIDENT_TYPE_NAMES = {
    "01": "사망사고",
    "02": "중상사고",
    "03": "경상사고",
    "04": "부상신고사고",
}


def parse_period_bound(value: Any, *, is_end: bool) -> pd.Timestamp:
    """입력 정밀도에 맞춰 시작/종료 경계를 해석합니다."""
    if isinstance(value, pd.Timestamp):
        timestamp = value
    else:
        text = str(value).strip()
        normalized = text.replace(".", "-").replace("/", "-")

        if re.fullmatch(r"\d{4}", normalized):
            period = pd.Period(normalized, freq="Y")
            timestamp = period.end_time if is_end else period.start_time
        elif re.fullmatch(r"\d{4}-\d{1,2}", normalized):
            period = pd.Period(normalized, freq="M")
            timestamp = period.end_time if is_end else period.start_time
        elif re.fullmatch(r"\d{4}-\d{1,2}-\d{1,2}", normalized):
            timestamp = pd.Timestamp(normalized).normalize()
            if is_end:
                timestamp += pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)
        else:
            timestamp = pd.Timestamp(text)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_localize(None)
    return timestamp


def normalize_legal_dong_code(value: Any) -> str:
    code = str(value).strip().replace(" ", "")
    if code.endswith("%"):
        code = code[:-1]
    if not re.fullmatch(r"\d{2,10}", code):
        raise ValueError("LEGAL_DONG_CODE는 % 없이 숫자 2~10자리로 입력하세요. 예: '11', '4511', '45113'")
    return code


def normalize_accident_types(values: Any) -> list[str]:
    if isinstance(values, str):
        pieces = re.split(r"[,;\s]+", values.strip())
    else:
        pieces = list(values)

    normalized: list[str] = []
    for value in pieces:
        code = str(value).strip()
        if not code:
            continue
        if code.isdigit():
            code = code.zfill(2)
        if code not in ACCIDENT_TYPE_NAMES:
            raise ValueError(f"지원하지 않는 사고 코드입니다: {code!r}. 01, 02, 03, 04 중에서 선택하세요.")
        if code not in normalized:
            normalized.append(code)

    if not normalized:
        raise ValueError("ACCIDENT_TYPES에 사고 코드를 하나 이상 지정하세요.")
    return normalized


start_bound = parse_period_bound(START_DATETIME, is_end=False)
end_bound = parse_period_bound(END_DATETIME, is_end=True)
if end_bound < start_bound:
    raise ValueError("END_DATETIME은 START_DATETIME보다 빠를 수 없습니다.")

legal_dong_code = normalize_legal_dong_code(LEGAL_DONG_CODE)
accident_type_codes = normalize_accident_types(ACCIDENT_TYPES)
request_accident_types = ",".join(accident_type_codes)

if len(legal_dong_code) not in {2, 4, 5, 10}:
    print("주의: 흔히 쓰는 법정동 코드 길이(2, 4, 5, 10자리)가 아닙니다. TAAS 요청 Payload의 값을 확인하세요.")

settings_summary = pd.DataFrame(
    {
        "항목": ["시작 일시", "종료 일시", "서버 조회 연도", "법정동 코드", "사고 유형"],
        "값": [
            str(start_bound),
            str(end_bound),
            f"{start_bound.year} ~ {end_bound.year}",
            legal_dong_code,
            ", ".join(f"{code}({ACCIDENT_TYPE_NAMES[code]})" for code in accident_type_codes),
        ],
    }
)
display(settings_summary)

,항목,값
0,시작 일시,2020-01-01 00:00:00
1,종료 일시,2024-12-31 23:59:59
2,서버 조회 연도,2020 ~ 2024
3,법정동 코드,2818510600
4,사고 유형,"01(사망사고), 02(중상사고), 03(경상사고), 04(부상신고사고)"


## 보안값 입력

아래 셀을 실행하면 입력창이 두 번 나타납니다. 개발자 도구에서 복사한 값을 붙여넣고 Enter를 누르세요.

- 첫 번째: `TAASJSESSIONID`
- 두 번째: `x-csrf-token`

입력 내용은 화면에 보이지 않고 노트북 파일에도 기록되지 않습니다. 커널을 재시작하면 다시 입력해야 합니다.

In [5]:
def extract_session_id(raw_value: str) -> str:
    text = raw_value.strip()
    match = re.search(r"TAASJSESSIONID=([^;\s]+)", text, flags=re.IGNORECASE)
    return (match.group(1) if match else text).strip().strip("'\"")


def extract_csrf_token(raw_value: str) -> str:
    text = re.sub(
        r"^\s*x-csrf-token\s*:\s*",
        "",
        raw_value.strip(),
        flags=re.IGNORECASE,
    )
    return text.strip().strip("'\"")


TAAS_JSESSION_ID = extract_session_id(getpass("TAASJSESSIONID 값을 붙여넣으세요: "))
X_CSRF_TOKEN = extract_csrf_token(getpass("x-csrf-token 값을 붙여넣으세요: "))

if not TAAS_JSESSION_ID or not X_CSRF_TOKEN:
    raise ValueError("두 값을 모두 입력해야 합니다. 개발자 도구 안내를 다시 확인하세요.")

print("보안값 입력 완료. 값은 출력하거나 파일로 저장하지 않습니다.")

보안값 입력 완료. 값은 출력하거나 파일로 저장하지 않습니다.


## TAAS 요청 함수

브라우저 헤더를 전부 복사할 필요는 없습니다. 요청에 필요한 최소 헤더와 세션 쿠키만 사용합니다. `Content-Length`, `Host`, `Accept-Encoding` 같은 값은 `requests`가 자동으로 계산합니다.

응답은 글에서 제시한 `resultValue.accidentInfoList` 구조를 확인한 뒤 DataFrame으로 변환합니다. 구조가 바뀌었거나 세션이 만료되면 조용히 빈 CSV를 만들지 않고, 원인을 확인할 수 있는 오류를 냅니다.

In [7]:
TAAS_ENDPOINT = "https://taas.koroad.or.kr/gis/srh/ash/selectAccidentInfo.do"
TAAS_REFERER = "https://taas.koroad.or.kr/gis/mcm/mcl/initMap.do?menuId=GIS_GMP_STS_RSN"


class TAASRequestError(RuntimeError):
    pass


def build_payload() -> dict[str, Any]:
    return {
        "searchType": "00",
        "pageIndex": 1,
        "zoneYn": False,
        "engnCode": "00",
        "startAcdntYear": str(start_bound.year),
        "endAcdntYear": str(end_bound.year),
        "legaldongCode": f"{legal_dong_code}%",
        "acdntGaeCode": request_accident_types,
    }


def response_excerpt(text: str, limit: int = 300) -> str:
    return re.sub(r"\s+", " ", text).strip()[:limit]


def fetch_taas_accidents(
    jsession_id: str,
    csrf_token: str,
) -> tuple[dict[str, Any], pd.DataFrame, int | None]:
    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Content-Type": "application/json;charset=UTF-8",
        "Origin": "https://taas.koroad.or.kr",
        "Referer": TAAS_REFERER,
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0 Safari/537.36"
        ),
        "X-CSRF-TOKEN": csrf_token,
        "X-Requested-With": "XMLHttpRequest",
    }

    try:
        with requests.Session() as session:
            session.cookies.set("TAASJSESSIONID", jsession_id)
            response = session.post(
                TAAS_ENDPOINT,
                json=build_payload(),
                headers=headers,
                timeout=(15, REQUEST_TIMEOUT_SECONDS),
            )
    except requests.Timeout as exc:
        raise TAASRequestError(
            "요청 시간이 초과되었습니다. 더 작은 지역/짧은 연도로 시험하거나 REQUEST_TIMEOUT_SECONDS를 늘리세요."
        ) from exc
    except requests.RequestException as exc:
        raise TAASRequestError(f"TAAS 연결에 실패했습니다: {exc}") from exc

    if response.status_code != 200:
        if response.status_code in {401, 403}:
            hint = "세션 또는 CSRF 토큰이 만료되었을 가능성이 큽니다. 개발자 도구에서 새 값을 가져오세요."
        elif response.status_code >= 500:
            hint = "세션 만료, 검색조건 오류 또는 TAAS 서버 오류일 수 있습니다. 작은 조건으로 TAAS 화면 조회부터 다시 시험하세요."
        else:
            hint = "TAAS 화면의 실제 요청 Payload와 사용자 설정을 비교하세요."
        raise TAASRequestError(
            f"TAAS 요청 실패 (HTTP {response.status_code}). {hint} "
            f"응답 일부: {response_excerpt(response.text)!r}"
        )

    try:
        body = response.json()
    except requests.JSONDecodeError as exc:
        raise TAASRequestError(
            "HTTP 200 응답이지만 JSON이 아닙니다. 세션 만료로 HTML이 반환되었거나 TAAS 화면이 변경되었을 수 있습니다. "
            f"응답 일부: {response_excerpt(response.text)!r}"
        ) from exc

    if not isinstance(body, dict):
        raise TAASRequestError(f"예상하지 못한 최상위 응답 형식입니다: {type(body).__name__}")

    result_value = body.get("resultValue")
    if not isinstance(result_value, dict):
        raise TAASRequestError(
            "응답에 resultValue 객체가 없습니다. "
            f"현재 최상위 키: {list(body)[:20]}"
        )

    accident_list = result_value.get("accidentInfoList")
    if not isinstance(accident_list, list):
        raise TAASRequestError(
            "응답에 resultValue.accidentInfoList 배열이 없습니다. "
            f"현재 resultValue 키: {list(result_value)[:30]}"
        )

    frame = pd.json_normalize(accident_list, sep=".")

    reported_total: int | None = None
    for key in ("totalCount", "totalCnt", "totCnt", "accidentInfoTotalCount"):
        value = result_value.get(key)
        if value is not None:
            try:
                reported_total = int(value)
            except (TypeError, ValueError):
                pass
            break

    return body, frame, reported_total

In [8]:
raw_response, accident_df, reported_total = fetch_taas_accidents(
    TAAS_JSESSION_ID,
    X_CSRF_TOKEN,
)

print(f"TAAS 응답 행 수: {len(accident_df):,}")
if reported_total is not None:
    print(f"응답이 알린 전체 건수: {reported_total:,}")
    if reported_total > len(accident_df):
        print("주의: 전체 건수보다 받은 행이 적습니다. TAAS에 페이지 제한이 생겼을 수 있으므로 결과를 검토하세요.")

if accident_df.empty:
    print("조회 결과가 0건입니다. 연도, 사고 유형, 법정동 코드를 확인하세요.")
else:
    print("응답 열:")
    print(", ".join(map(str, accident_df.columns)))
    with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
        display(accident_df.head(5))

TAAS 응답 행 수: 1,202
응답 열:
searchConditionText, searchCondition, searchKeyword, pageIndex, pageUnit, recordCountPerPage, zoneYn, acdnt_no, engn_code, acdnt_year, acdnt_dd_dc, dfk_code, dfk_dc, tmzon_div_1_code, tmzon_div_1_dc, occrrnc_time_code, occrrnc_time_dc, legaldong_code, legaldong_name, acdnt_hcode, acdnt_hdc, acdnt_mcode, acdnt_mdc, acdnt_code, acdnt_dc, lrg_violt_1_code, lrg_violt_1_dc, x_crdnt, y_crdnt, wether_sttus_code, wether_sttus_dc, road_stle_code, road_stle_dc, hhdgw_at, acdnt_gae_code, acdnt_gae_dc, dprs_cnt, sep_cnt, slp_cnt, inj_aplcnt_cnt, wrngdo_vhcle_asort_hcode, wrngdo_vhcle_asort_hdc, wrngdo_vhcle_asort_code, wrngdo_vhcle_asort_dc, injury_dgree_1_hcode, injury_dgree_1_hdc, injury_dgree_1_code, injury_dgree_1_dc, acdnt_age_1_code, acdnt_age_1_dc, sexdstn_div_1_code, sexdstn_div_1_dc, dmge_vhcle_asort_hcode, dmge_vhcle_asort_hdc, dmge_vhcle_asort_code, dmge_vhcle_asort_dc, injury_dgree_2_hcode, injury_dgree_2_hdc, injury_dgree_2_code, injury_dgree_2_dc, acdnt_age_2

,searchConditionText,searchCondition,searchKeyword,pageIndex,pageUnit,recordCountPerPage,zoneYn,acdnt_no,engn_code,acdnt_year,acdnt_dd_dc,dfk_code,dfk_dc,tmzon_div_1_code,tmzon_div_1_dc,occrrnc_time_code,occrrnc_time_dc,legaldong_code,legaldong_name,acdnt_hcode,acdnt_hdc,acdnt_mcode,acdnt_mdc,acdnt_code,acdnt_dc,lrg_violt_1_code,lrg_violt_1_dc,x_crdnt,y_crdnt,wether_sttus_code,wether_sttus_dc,road_stle_code,road_stle_dc,hhdgw_at,acdnt_gae_code,acdnt_gae_dc,dprs_cnt,sep_cnt,slp_cnt,inj_aplcnt_cnt,wrngdo_vhcle_asort_hcode,wrngdo_vhcle_asort_hdc,wrngdo_vhcle_asort_code,wrngdo_vhcle_asort_dc,injury_dgree_1_hcode,injury_dgree_1_hdc,injury_dgree_1_code,injury_dgree_1_dc,acdnt_age_1_code,acdnt_age_1_dc,sexdstn_div_1_code,sexdstn_div_1_dc,dmge_vhcle_asort_hcode,dmge_vhcle_asort_hdc,dmge_vhcle_asort_code,dmge_vhcle_asort_dc,injury_dgree_2_hcode,injury_dgree_2_hdc,injury_dgree_2_code,injury_dgree_2_dc,acdnt_age_2_code,acdnt_age_2_dc,sexdstn_div_2_code,sexdstn_div_2_dc,bdy_injury_part_1_code,bdy_injury_part_1_dc,bdy_injury_part_2_code,bdy_injury_part_2_dc,spt_otlnmap_at,rdse_sttus_code,rdse_sttus_dc,acdnt_pos,acdnt_des,road_div,road_no,route_nm,otn_acdnt_no,city_idt_code,city_idt_dc,xCrdnt,yCrdnt,acdnt_frm_lv1,acdnt_frm_lv2,acdnt_frm_lv3,acdnt_sta_lv1,acdnt_sta_lv2,rde_id,rde_nm,serial_no,rn
0,,,,1,3,10,False,2020010200100423,00,2020,2020년 1월,4,목요일,02,야간,18,18시,2818510600,인천광역시 연수구,200,차대차,220,측면충돌,220,차대차 - 충돌,06,안전운전불이행,927328,1931664,01,맑음,299,단일로 - 기타,N,03,경상사고,0,0,1,0,100,원동기 자전거 이상,110,승용,300,상해없음,300,상해없음,29,21-30세,1,남,100,원동기 자전거 이상,110,승용,200,부상,220,경상,29,21-30세,1,남,00,상해없음,98,기타,없음,01,건조,세브란스병원 예정지 앞 노상,NaN,104,0,송도바이오대로,20200001152309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020000756,None
1,,,,1,3,10,False,2020010200100489,00,2020,2020년 1월,4,목요일,02,야간,20,20시,2818510600,인천광역시 연수구,200,차대차,220,측면충돌,220,차대차 - 충돌,06,안전운전불이행,923427,1933527,01,맑음,110,교차로 - 교차로안,N,03,경상사고,0,0,1,0,100,원동기 자전거 이상,130,화물,300,상해없음,300,상해없음,54,51-60세,1,남,100,원동기 자전거 이상,150,이륜,200,부상,220,경상,28,21-30세,1,남,00,상해없음,06,뒷면 뒷목,없음,01,건조,ibs 타워 사거리,NaN,104,0,센트럴로,20200000382309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020000820,None
2,,,,1,3,10,False,2020010300100507,00,2020,2020년 1월,5,금요일,02,야간,19,19시,2818510600,인천광역시 연수구,200,차대차,299,기타,299,차대차 - 기타,06,안전운전불이행,924360,1931741,01,맑음,299,단일로 - 기타,N,03,경상사고,0,0,1,0,998,기타,998,기타불명,998,기타,998,기타불명,999,기타불명,8,기타불명,100,원동기 자전거 이상,160,원동기,200,부상,220,경상,20,13-20세,1,남,00,상해없음,08,뒷면 허리,없음,01,건조,대림아파트 인근 노상,NaN,NaN,NaN,NaN,20200000392309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020001382,None
3,,,,1,3,10,False,2020010400100161,00,2020,2020년 1월,6,토요일,01,주간,10,10시,2818510600,인천광역시 연수구,200,차대차,220,측면충돌,220,차대차 - 충돌,02,신호위반,925473,1929927,01,맑음,110,교차로 - 교차로안,N,03,경상사고,0,0,1,0,300,건설기계,300,건설기계,300,상해없음,300,상해없음,60,51-60세,1,남,100,원동기 자전거 이상,110,승용,200,부상,220,경상,52,51-60세,2,여,00,상해없음,06,뒷면 뒷목,없음,01,건조,송도바이오로직스 앞 사거리,NaN,104,0,송도문화로120번길,20200000462309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020001650,None
4,,,,1,3,10,False,2020010400100433,00,2020,2020년 1월,6,토요일,02,야간,19,19시,2818510600,인천광역시 연수구,200,차대차,220,측면충돌,220,차대차 - 충돌,98,기타,925201,1931579,01,맑음,299,단일로 - 기타,N,03,경상사고,0,0,1,0,100,원동기 자전거 이상,110,승용,300,상해없음,300,상해없음,62,61-64세,1,남,100,원동기 자전거 이상,110,승용,200,부상,220,경상,33,31-40세,2,여,00,상해없음,98,기타,없음,01,건조,송도홈플러스 앞 사거리,NaN,104,0,송도국제대로,20200000832309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020001905,None


## 상세 시작·종료 일시 필터

TAAS 서버에는 시작·종료 **연도**를 보냈습니다. 설정한 기간이 연도 전체보다 좁을 때만 아래 코드가 사고일시 열을 자동으로 찾아 로컬 필터를 적용합니다.

자동 탐지가 실패하면 직전 셀에 출력된 열 이름을 보고 사용자 설정 셀의 `DATETIME_COLUMN` 또는 `DATE_COLUMN`/`TIME_COLUMN`을 지정한 뒤, 설정 검증 셀부터 다시 실행하세요. 사고일시 열을 찾지 못한 상태에서 상세 기간을 무시하고 저장하지 않도록 오류를 내는 것이 정상 동작입니다.

In [9]:
DATE_COLUMN_CANDIDATES = [
    "acdntDttm",
    "acdntDt",
    "acdntDate",
    "acdntYmd",
    "accidentDatetime",
    "accidentDate",
    "occrrncDttm",
    "occrrncDt",
    "occrrncYmd",
    "사고일시",
    "사고일자",
    "발생일시",
    "발생일자",
]
TIME_COLUMN_CANDIDATES = [
    "acdntTime",
    "acdntHour",
    "accidentTime",
    "occrrncTime",
    "사고시간",
    "발생시간",
]


def normalize_column_name(value: Any) -> str:
    return re.sub(r"[^0-9a-z가-힣]", "", str(value).lower())


def resolve_column(
    frame: pd.DataFrame,
    requested: str | None,
    candidates: list[str],
    *,
    heuristic_markers: tuple[str, ...],
) -> str | None:
    normalized_to_original = {normalize_column_name(column): column for column in frame.columns}

    if requested:
        resolved = normalized_to_original.get(normalize_column_name(requested))
        if resolved is None:
            raise KeyError(
                f"지정한 열 {requested!r}을 찾지 못했습니다. 현재 열: {list(frame.columns)}"
            )
        return str(resolved)

    for candidate in candidates:
        resolved = normalized_to_original.get(normalize_column_name(candidate))
        if resolved is not None:
            return str(resolved)

    for column in frame.columns:
        normalized = normalize_column_name(column)
        if any(marker in normalized for marker in heuristic_markers):
            return str(column)
    return None


def parse_datetime_series(series: pd.Series) -> pd.Series:
    text = series.astype("string").str.strip()
    text = text.str.replace(
        r"(\d{4})\s*년\s*(\d{1,2})\s*월\s*(\d{1,2})\s*일",
        r"\1-\2-\3",
        regex=True,
    )
    text = text.str.replace(r"(\d{1,2})\s*시", r"\1:", regex=True)
    text = text.str.replace(r"(\d{1,2})\s*분", r"\1", regex=True)
    text = text.str.replace(r":(?=\s*$)", ":00", regex=True)

    try:
        parsed = pd.to_datetime(text, errors="coerce", format="mixed")
    except (TypeError, ValueError):
        parsed = pd.to_datetime(text, errors="coerce")
    if parsed.notna().sum() == 0 and text.notna().any():
        # pandas 2.0 이전 버전에서는 format='mixed'가 지원되지 않을 수 있습니다.
        parsed = pd.to_datetime(text, errors="coerce")
    return parsed


def parse_time_value(value: Any) -> Any:
    if pd.isna(value):
        return pd.NaT

    text = str(value).strip()
    if not text:
        return pd.NaT

    korean = re.fullmatch(
        r"(\d{1,2})\s*시(?:\s*(\d{1,2})\s*분)?(?:\s*(\d{1,2})\s*초)?",
        text,
    )
    colon = re.fullmatch(r"(\d{1,2}):(\d{1,2})(?::(\d{1,2}))?", text)

    if korean or colon:
        match = korean or colon
        assert match is not None
        hour = int(match.group(1))
        minute = int(match.group(2) or 0)
        second = int(match.group(3) or 0)
    else:
        digits = re.sub(r"\.0+$", "", text)
        if not digits.isdigit():
            return pd.NaT
        if len(digits) <= 2:
            hour, minute, second = int(digits), 0, 0
        elif len(digits) <= 4:
            digits = digits.zfill(4)
            hour, minute, second = int(digits[:2]), int(digits[2:]), 0
        elif len(digits) <= 6:
            digits = digits.zfill(6)
            hour, minute, second = int(digits[:2]), int(digits[2:4]), int(digits[4:])
        else:
            return pd.NaT

    if not (0 <= hour <= 23 and 0 <= minute <= 59 and 0 <= second <= 59):
        return pd.NaT
    return pd.Timedelta(hours=hour, minutes=minute, seconds=second)


server_range_start = pd.Timestamp(year=start_bound.year, month=1, day=1)
server_range_end = (
    pd.Timestamp(year=end_bound.year + 1, month=1, day=1) - pd.Timedelta(nanoseconds=1)
)
# 사용자가 초 단위로 12월 31일 23:59:59를 입력해도 연도 전체로 취급합니다.
server_range_end_at_second_precision = server_range_end.floor("s")
local_filter_needed = (
    start_bound > server_range_start
    or end_bound < server_range_end_at_second_precision
)

export_df = accident_df.copy()
datetime_source = None
filter_applied = False

if export_df.empty:
    print("응답이 0건이므로 상세 일시 필터를 건너뜁니다.")
elif not APPLY_EXACT_DATETIME_FILTER:
    print("APPLY_EXACT_DATETIME_FILTER=False: 서버에서 받은 연도 범위를 그대로 사용합니다.")
elif not local_filter_needed:
    print("설정 기간이 조회 연도 전체이므로 추가 일시 필터가 필요하지 않습니다.")
else:
    requested_date_column = DATETIME_COLUMN or DATE_COLUMN
    date_column = resolve_column(
        export_df,
        requested_date_column,
        DATE_COLUMN_CANDIDATES,
        heuristic_markers=("date", "dttm", "ymd", "일시", "일자"),
    )
    if date_column is None:
        raise RuntimeError(
            "상세 기간을 적용할 사고일시 열을 찾지 못했습니다. 직전 셀의 응답 열을 확인하고 "
            "DATETIME_COLUMN 또는 DATE_COLUMN/TIME_COLUMN을 지정하세요."
        )

    parsed_datetime = parse_datetime_series(export_df[date_column])
    datetime_source = date_column

    # DATETIME_COLUMN을 직접 지정했다면 그 열 자체에 시간이 있다고 간주합니다.
    # 그렇지 않으면 별도 시간 열을 자동 탐지하거나 TIME_COLUMN 설정을 사용합니다.
    time_column = None
    if DATETIME_COLUMN is None:
        time_column = resolve_column(
            export_df,
            TIME_COLUMN,
            TIME_COLUMN_CANDIDATES,
            heuristic_markers=("time", "hour", "시간"),
        )

    if time_column is not None:
        time_delta = pd.to_timedelta(export_df[time_column].map(parse_time_value))
        parsed_datetime = parsed_datetime.dt.normalize() + time_delta
        datetime_source = f"{date_column} + {time_column}"

    parsed_count = int(parsed_datetime.notna().sum())
    if parsed_count == 0:
        raise RuntimeError(
            f"{datetime_source!r} 열을 날짜/시간으로 해석하지 못했습니다. "
            "원본 값 형식을 확인하고 열 설정을 수정하세요."
        )

    export_df["accident_datetime_parsed"] = parsed_datetime
    unparsed_count = len(export_df) - parsed_count
    if unparsed_count:
        print(f"주의: 사고일시를 해석하지 못한 {unparsed_count:,}행은 상세 기간 결과에서 제외됩니다.")

    before_count = len(export_df)
    export_df = export_df.loc[
        export_df["accident_datetime_parsed"].between(
            start_bound,
            end_bound,
            inclusive="both",
        )
    ].copy()
    filter_applied = True
    print(f"상세 일시 필터 열: {datetime_source}")
    print(f"상세 일시 필터 결과: {before_count:,}행 → {len(export_df):,}행")

if not export_df.empty:
    with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
        display(export_df.head(5))

설정 기간이 조회 연도 전체이므로 추가 일시 필터가 필요하지 않습니다.


,searchConditionText,searchCondition,searchKeyword,pageIndex,pageUnit,recordCountPerPage,zoneYn,acdnt_no,engn_code,acdnt_year,acdnt_dd_dc,dfk_code,dfk_dc,tmzon_div_1_code,tmzon_div_1_dc,occrrnc_time_code,occrrnc_time_dc,legaldong_code,legaldong_name,acdnt_hcode,acdnt_hdc,acdnt_mcode,acdnt_mdc,acdnt_code,acdnt_dc,lrg_violt_1_code,lrg_violt_1_dc,x_crdnt,y_crdnt,wether_sttus_code,wether_sttus_dc,road_stle_code,road_stle_dc,hhdgw_at,acdnt_gae_code,acdnt_gae_dc,dprs_cnt,sep_cnt,slp_cnt,inj_aplcnt_cnt,wrngdo_vhcle_asort_hcode,wrngdo_vhcle_asort_hdc,wrngdo_vhcle_asort_code,wrngdo_vhcle_asort_dc,injury_dgree_1_hcode,injury_dgree_1_hdc,injury_dgree_1_code,injury_dgree_1_dc,acdnt_age_1_code,acdnt_age_1_dc,sexdstn_div_1_code,sexdstn_div_1_dc,dmge_vhcle_asort_hcode,dmge_vhcle_asort_hdc,dmge_vhcle_asort_code,dmge_vhcle_asort_dc,injury_dgree_2_hcode,injury_dgree_2_hdc,injury_dgree_2_code,injury_dgree_2_dc,acdnt_age_2_code,acdnt_age_2_dc,sexdstn_div_2_code,sexdstn_div_2_dc,bdy_injury_part_1_code,bdy_injury_part_1_dc,bdy_injury_part_2_code,bdy_injury_part_2_dc,spt_otlnmap_at,rdse_sttus_code,rdse_sttus_dc,acdnt_pos,acdnt_des,road_div,road_no,route_nm,otn_acdnt_no,city_idt_code,city_idt_dc,xCrdnt,yCrdnt,acdnt_frm_lv1,acdnt_frm_lv2,acdnt_frm_lv3,acdnt_sta_lv1,acdnt_sta_lv2,rde_id,rde_nm,serial_no,rn
0,,,,1,3,10,False,2020010200100423,00,2020,2020년 1월,4,목요일,02,야간,18,18시,2818510600,인천광역시 연수구,200,차대차,220,측면충돌,220,차대차 - 충돌,06,안전운전불이행,927328,1931664,01,맑음,299,단일로 - 기타,N,03,경상사고,0,0,1,0,100,원동기 자전거 이상,110,승용,300,상해없음,300,상해없음,29,21-30세,1,남,100,원동기 자전거 이상,110,승용,200,부상,220,경상,29,21-30세,1,남,00,상해없음,98,기타,없음,01,건조,세브란스병원 예정지 앞 노상,NaN,104,0,송도바이오대로,20200001152309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020000756,None
1,,,,1,3,10,False,2020010200100489,00,2020,2020년 1월,4,목요일,02,야간,20,20시,2818510600,인천광역시 연수구,200,차대차,220,측면충돌,220,차대차 - 충돌,06,안전운전불이행,923427,1933527,01,맑음,110,교차로 - 교차로안,N,03,경상사고,0,0,1,0,100,원동기 자전거 이상,130,화물,300,상해없음,300,상해없음,54,51-60세,1,남,100,원동기 자전거 이상,150,이륜,200,부상,220,경상,28,21-30세,1,남,00,상해없음,06,뒷면 뒷목,없음,01,건조,ibs 타워 사거리,NaN,104,0,센트럴로,20200000382309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020000820,None
2,,,,1,3,10,False,2020010300100507,00,2020,2020년 1월,5,금요일,02,야간,19,19시,2818510600,인천광역시 연수구,200,차대차,299,기타,299,차대차 - 기타,06,안전운전불이행,924360,1931741,01,맑음,299,단일로 - 기타,N,03,경상사고,0,0,1,0,998,기타,998,기타불명,998,기타,998,기타불명,999,기타불명,8,기타불명,100,원동기 자전거 이상,160,원동기,200,부상,220,경상,20,13-20세,1,남,00,상해없음,08,뒷면 허리,없음,01,건조,대림아파트 인근 노상,NaN,NaN,NaN,NaN,20200000392309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020001382,None
3,,,,1,3,10,False,2020010400100161,00,2020,2020년 1월,6,토요일,01,주간,10,10시,2818510600,인천광역시 연수구,200,차대차,220,측면충돌,220,차대차 - 충돌,02,신호위반,925473,1929927,01,맑음,110,교차로 - 교차로안,N,03,경상사고,0,0,1,0,300,건설기계,300,건설기계,300,상해없음,300,상해없음,60,51-60세,1,남,100,원동기 자전거 이상,110,승용,200,부상,220,경상,52,51-60세,2,여,00,상해없음,06,뒷면 뒷목,없음,01,건조,송도바이오로직스 앞 사거리,NaN,104,0,송도문화로120번길,20200000462309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020001650,None
4,,,,1,3,10,False,2020010400100433,00,2020,2020년 1월,6,토요일,02,야간,19,19시,2818510600,인천광역시 연수구,200,차대차,220,측면충돌,220,차대차 - 충돌,98,기타,925201,1931579,01,맑음,299,단일로 - 기타,N,03,경상사고,0,0,1,0,100,원동기 자전거 이상,110,승용,300,상해없음,300,상해없음,62,61-64세,1,남,100,원동기 자전거 이상,110,승용,200,부상,220,경상,33,31-40세,2,여,00,상해없음,98,기타,없음,01,건조,송도홈플러스 앞 사거리,NaN,104,0,송도국제대로,20200000832309,None,None,0.0,0.0,None,None,None,None,None,None,None,2020001905,None


## JSON·CSV·메타데이터 저장

- `*_raw.json`: TAAS가 보낸 원본 응답 전체
- `*.csv`: 상세 시작·종료 일시 필터를 반영한 사고 목록 (`utf-8-sig`)
- `*_metadata.json`: 실행 조건, 행 수, 사용 열을 기록한 재현성 메타데이터

세 파일 어디에도 세션 ID와 CSRF 토큰은 저장하지 않습니다.

In [10]:
output_directory = Path(OUTPUT_DIRECTORY).expanduser().resolve()
output_directory.mkdir(parents=True, exist_ok=True)

period_tag = f"{start_bound:%Y%m%dT%H%M%S}_{end_bound:%Y%m%dT%H%M%S}"
file_stem = f"{OUTPUT_BASENAME}_{period_tag}_dong{legal_dong_code}"
raw_json_path = output_directory / f"{file_stem}_raw.json"
csv_path = output_directory / f"{file_stem}.csv"
metadata_path = output_directory / f"{file_stem}_metadata.json"

raw_json_path.write_text(
    json.dumps(raw_response, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
export_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

metadata = {
    "source": "TAAS 교통사고분석시스템",
    "source_url": TAAS_REFERER,
    "reference_blog": "https://yvelkram.tistory.com/2",
    "endpoint": TAAS_ENDPOINT,
    "start_datetime": start_bound.isoformat(),
    "end_datetime": end_bound.isoformat(),
    "legal_dong_code": legal_dong_code,
    "accident_types": {code: ACCIDENT_TYPE_NAMES[code] for code in accident_type_codes},
    "request_payload": build_payload(),
    "raw_row_count": len(accident_df),
    "export_row_count": len(export_df),
    "exact_datetime_filter_requested": APPLY_EXACT_DATETIME_FILTER,
    "exact_datetime_filter_applied": filter_applied,
    "datetime_source_column": datetime_source,
    "columns": list(map(str, export_df.columns)),
    "coordinate_reference_system_note": "블로그 설명 기준 EPSG:5179; 실제 좌표 열과 범위를 확인할 것",
}
metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("저장 완료")
print(f"- 원본 JSON: {raw_json_path}")
print(f"- 분석용 CSV: {csv_path}")
print(f"- 메타데이터: {metadata_path}")

저장 완료
- 원본 JSON: C:\Users\USER\Documents\교통 관련 논문 작성\notebook\taas_output\taas_accidents_20200101T000000_20241231T235959_dong2818510600_raw.json
- 분석용 CSV: C:\Users\USER\Documents\교통 관련 논문 작성\notebook\taas_output\taas_accidents_20200101T000000_20241231T235959_dong2818510600.csv
- 메타데이터: C:\Users\USER\Documents\교통 관련 논문 작성\notebook\taas_output\taas_accidents_20200101T000000_20241231T235959_dong2818510600_metadata.json


## QGIS에서 열기 및 문제 해결

### QGIS

블로그 설명 기준 TAAS 위치 좌표계는 `EPSG:5179`입니다. CSV를 **구분된 텍스트 레이어 추가**로 불러온 뒤 응답의 X/Y 좌표 열을 지정하고 CRS를 `EPSG:5179`로 선택하세요. TAAS 응답 구조가 바뀔 수 있으므로 좌표 열 이름과 값의 범위를 먼저 확인하세요. `geomJson`처럼 중첩된 값은 `pd.json_normalize()` 때문에 점(`.`)이 포함된 열 이름으로 펼쳐질 수 있습니다.

### 자주 발생하는 오류

- **HTTP 401/403**: 세션 ID 또는 CSRF 토큰이 만료된 경우가 많습니다. TAAS 화면을 새로 열어 실제 조회 후 두 값을 다시 입력하세요.
- **HTTP 500**: 만료된 세션, 잘못된 검색값, 너무 큰 요청 또는 일시적 서버 오류일 수 있습니다. TAAS 화면에서 같은 조건이 되는지 확인하고 작은 지역·한 해로 먼저 시험하세요.
- **HTTP 200인데 JSON 오류**: 로그인/안내 HTML이 반환되었을 수 있습니다. 세션을 새로 가져오세요.
- **결과 0건**: 조회 연도, 법정동 코드, 사고 코드를 확인하세요. `LEGAL_DONG_CODE`에는 `%`를 적지 않습니다.
- **사고일시 열 자동 탐지 실패**: 출력된 열 이름과 샘플을 확인하고 `DATETIME_COLUMN` 또는 `DATE_COLUMN`/`TIME_COLUMN`을 직접 지정하세요.
- **받은 행 수가 전체 건수보다 적음**: TAAS의 페이지 처리 방식이 바뀌었을 수 있습니다. 이 노트북은 누락 가능성을 경고하며, 확인 없이 완전한 데이터라고 간주하면 안 됩니다.

Colab에서 실행했다면 왼쪽 파일 패널의 `taas_output` 폴더에서 결과를 내려받을 수 있습니다. 로컬 Jupyter라면 이 노트북이 실행된 작업 폴더 아래에 저장됩니다.